# FINAL 3 TINGKAT: post-editing leksikon Krama → Krama Alus

Pertanyaan: apakah output model demo v2 yang **diminta Krama Alus tetapi keluar Krama** bisa diperbaiki dengan
mengganti kata Krama memakai leksikon Krama → Krama Alus (`data/lexicon/krama_to_krama_alus.csv`, ditambang dari train)?

Dasar: post-editing berbasis aturan menaikkan akurasi formality tanpa menurunkan BLEU (Zhang dkk., IWSLT 2022;
Feely dkk., WAT 2019). Leksikon yang sama **merugikan sebagai augmentasi** (`final_3lv_merge`, varian `aug`), tetapi di
valid substitusinya menaikkan chrF++ Krama → Krama Alus dari 55,24 ke 70,66 (`scripts/build_lexicon_augment.ipynb`).

Desain:
- **Tanpa training MT.** Input = prediksi tersimpan model demo v2 (`kaggle/final_3lv_v2/output/`): `rerank2.0` (demo)
  dan `beam4` (tanpa rerank).
- Post-editing hanya untuk permintaan **Krama Alus**. Mode:
  `none` · `all` (semua diedit) · `trig` (diedit jika reranker menilai output bukan Krama Alus) ·
  `gain` (diedit jika P(Krama Alus) reranker naik setelah diedit) · `trig_gain` (keduanya).
- Pemicu memakai **reranker** (`w11wo/javanese-roberta-small`), bukan juri (`w11wo/javanese-distilbert-small`),
  supaya ketepatan tingkat tidak naik palsu. Keduanya dilatih ulang di train split dengan resep v2.
- **Mode dipilih di valid**, bukan test. Prediksi valid model tidak tersimpan, jadi dipakai simulasi dari referensi
  asli: referensi Krama = output yang meleset (harus diperbaiki), referensi Krama Alus = output yang sudah benar
  (tidak boleh rusak). Skor = chrF++ + 30 × ketepatan tingkat (sama dengan pemilihan di v2).
- Juri yang sama menilai baseline dan hasil post-editing, jadi perbandingan di dalam run ini adil. Angka ketepatan
  tidak bisa dibandingkan langsung dengan kernel lain (juri dilatih ulang tiap run).
- Bisa dijalankan lokal (CPU, ~10 menit) atau di Kaggle.


In [1]:
import os, re, json, random, time, glob
import numpy as np, pandas as pd, torch, sacrebleu
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

KAGGLE = os.path.exists("/kaggle/input")
ROOT = os.path.abspath("../..")                  # notebook ada di kaggle/final_3lv_postedit/
OUT = "/kaggle/working" if KAGGLE else "output"
os.makedirs(OUT, exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"
if dev == "cpu":
    torch.set_num_threads(os.cpu_count())

LEVELS = ["ngoko", "krama", "krama_alus"]
LID = {l: i for i, l in enumerate(LEVELS)}
KA = LID["krama_alus"]
SEED, MAXLEN, CTRL_W = 42, 64, 30.0
MODES = ["none", "trig_gain", "trig", "gain", "all"]   # urutan = prioritas saat skor seri (lebih sedikit edit dulu)
T0 = time.time()
log = lambda *a: print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)


def path(rel):
    """Lokal: relatif ke root repo. Kaggle: cari nama file di /kaggle/input."""
    if not KAGGLE:
        return os.path.join(ROOT, rel)
    return glob.glob(f"/kaggle/input/**/{os.path.basename(rel)}", recursive=True)[0]


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)


def _load3(n):
    d = pd.read_csv(path(f"data/unggah_ungguh/split/{n}.csv"))
    d = d[d["level"].isin(LEVELS)].copy()
    d["label"] = d["level"].map(LID)
    return d.reset_index(drop=True)


tr, va, te = (_load3(n) for n in ["train", "valid", "test"])
log("data", len(tr), len(va), len(te), "| device", dev)

bleu = lambda h, r: sacrebleu.corpus_bleu(h, [r]).score
chrf = lambda h, r: sacrebleu.corpus_chrf(h, [r], word_order=2).score


def boot_ci(h, r, n=1000, seed=0):
    rng = random.Random(seed); idx = range(len(h)); vals = []
    for _ in range(n):
        s = [rng.choice(idx) for _ in idx]
        vals.append(chrf([h[i] for i in s], [r[i] for i in s]))
    vals.sort()
    return [round(vals[int(.025 * n)], 2), round(vals[int(.975 * n)], 2)]


# ---- leksikon Krama -> Krama Alus (ditambang dari train split oleh scripts/build_lexicon_augment.ipynb)
lex = pd.read_csv(path("data/lexicon/krama_to_krama_alus.csv"))
lex = lex[lex["reliable"].astype(str).str.lower() == "true"]
SUB = dict(lex[["src", "tgt"]].values)
WORD = re.compile(r"[^\W\d_]+(?:[-'][^\W\d_]+)*")


def to_krama_alus(sentence):
    """Sama dengan scripts/build_lexicon_augment.ipynb: ganti kata Krama dengan padanan Krama Alus."""
    def rep(m):
        w = m.group(0); t = SUB.get(w.lower())
        if t is None:
            return w
        return t.capitalize() if w[0].isupper() else t
    return WORD.sub(rep, str(sentence))


log("leksikon andal:", len(SUB), "pasangan, contoh:", list(SUB.items())[:6])


C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[     0s] data 1395 175 176 | device cpu


[     0s] leksikon andal: 57 pasangan, contoh: [('sampeyan', 'panjenengan'), ('kula', 'adalem'), ('gadhah', 'kagungan'), ('anakipun', 'putranipun'), ('piyambakipun', 'panjenenganipun'), ('mangga', 'sumangga')]


In [2]:
# ------------------------------------------------------------------ classifiers (resep sama dengan final_3lv_v2)
def train_classifier(base, seed):
    ctok = AutoTokenizer.from_pretrained(base)
    cmod = AutoModelForSequenceClassification.from_pretrained(base, num_labels=len(LEVELS)).to(dev)
    seed_all(seed)
    opt = torch.optim.AdamW(cmod.parameters(), lr=5e-5)
    X, y = tr["javanese"].tolist(), tr["label"].tolist()
    for _ in range(4):
        cmod.train(); order = list(range(len(X))); random.shuffle(order)
        for i in range(0, len(order), 32):
            b = order[i:i + 32]
            enc = ctok([X[j] for j in b], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            loss = cmod(**enc, labels=torch.tensor([y[j] for j in b], device=dev)).loss
            opt.zero_grad(); loss.backward(); opt.step()
    cmod.eval()

    def logprobs(sents):
        out = []
        for i in range(0, len(sents), 64):
            enc = ctok(list(sents[i:i + 64]), return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            with torch.no_grad():
                out.append(F.log_softmax(cmod(**enc).logits.float(), -1).cpu())
        return torch.cat(out).numpy()

    info = {"base": base}
    for n, d in [("valid", va), ("test", te)]:
        p = logprobs(d["javanese"].tolist()).argmax(-1); y_ = d["label"].values
        info[f"acc_{n}"] = round(float((p == y_).mean()), 4)
        info[f"per_class_acc_{n}"] = {l: round(float((p[y_ == i] == i).mean()), 4) for i, l in enumerate(LEVELS)}
    log("classifier", info)
    return logprobs, info


judge_lp, judge_info = train_classifier("w11wo/javanese-distilbert-small", SEED)
rr_lp, rr_info = train_classifier("w11wo/javanese-roberta-small", 7)
classify = lambda s: judge_lp(list(s)).argmax(-1)


C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--w11wo--javanese-distilbert-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 20008.13it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: w11wo/javanese-distilbert-small
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[   427s] classifier {'base': 'w11wo/javanese-distilbert-small', 'acc_valid': 0.9543, 'per_class_acc_valid': {'ngoko': 0.9865, 'krama': 1.0, 'krama_alus': 0.7407}, 'acc_test': 0.9375, 'per_class_acc_test': {'ngoko': 0.9733, 'krama': 0.9865, 'krama_alus': 0.7037}}


... (3 outputs left out)

Loading weights:  32%|███▏      | 63/197 [00:00<00:00, 218.16it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 606.31it/s]


[transformers] RobertaForSequenceClassification LOAD REPORT from: w11wo/javanese-roberta-small
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[  1353s] classifier {'base': 'w11wo/javanese-roberta-small', 'acc_valid': 0.9714, 'per_class_acc_valid': {'ngoko': 0.9865, 'krama': 0.9865, 'krama_alus': 0.8889}, 'acc_test': 0.9602, 'per_class_acc_test': {'ngoko': 1.0, 'krama': 0.9865, 'krama_alus': 0.7778}}


In [3]:
# ------------------------------------------------------------------ post-editing
def postedit(outs, reqs, mode):
    """Edit hanya output yang diminta Krama Alus. Pemicu memakai RERANKER, bukan juri."""
    outs = list(outs); new = outs[:]
    idx = [i for i, l in enumerate(reqs) if l == "krama_alus"]
    st = {"diminta_krama_alus": len(idx), "bisa_diubah": 0, "diubah": 0}
    if mode == "none" or not idx:
        return new, st
    cand = [to_krama_alus(outs[i]) for i in idx]
    old, cnd = rr_lp([outs[i] for i in idx]), rr_lp(cand)
    for k, i in enumerate(idx):
        if cand[k] == outs[i]:
            continue
        st["bisa_diubah"] += 1
        trig = old[k].argmax() != KA
        gain = cnd[k, KA] > old[k, KA]
        if {"all": True, "trig": trig, "gain": gain, "trig_gain": trig and gain}[mode]:
            new[i] = cand[k]; st["diubah"] += 1
    return new, st


# ---- pemilihan mode di VALID (simulasi dari referensi asli)
sim = []
for _, g in va.groupby("group"):
    jv = dict(zip(g["level"], g["javanese"]))
    if "krama_alus" not in jv:
        continue
    if "krama" in jv and jv["krama"] != jv["krama_alus"]:
        sim.append({"output": jv["krama"], "ref": jv["krama_alus"], "jenis": "meleset (ref Krama)"})
    sim.append({"output": jv["krama_alus"], "ref": jv["krama_alus"], "jenis": "benar (ref Krama Alus)"})
sim = pd.DataFrame(sim)
log("simulasi valid:", sim["jenis"].value_counts().to_dict())

valid_sel = {}
for m in MODES:
    h, st = postedit(sim["output"], ["krama_alus"] * len(sim), m)
    acc = float((classify(h) == KA).mean())
    c = chrf(h, sim["ref"].tolist())
    sub = {}
    for j, d in sim.groupby("jenis"):
        hh = [h[i] for i in d.index]
        sub[j] = {"chrF++": round(chrf(hh, d["ref"].tolist()), 2),
                  "ketepatan_krama_alus": round(float((classify(hh) == KA).mean()), 4)}
    valid_sel[m] = {"score": round(c + CTRL_W * acc, 2), "chrF++": round(c, 2), "ketepatan_krama_alus": round(acc, 4),
                    "per_jenis": sub, **st}
    log("valid", m, valid_sel[m])
best_mode = max(MODES, key=lambda m: (valid_sel[m]["score"], -MODES.index(m)))
log("MODE TERPILIH (valid):", best_mode)


[  1353s] simulasi valid: {'benar (ref Krama Alus)': 27, 'meleset (ref Krama)': 26}


[  1354s] valid none {'score': 89.68, 'chrF++': 78.36, 'ketepatan_krama_alus': 0.3774, 'per_jenis': {'benar (ref Krama Alus)': {'chrF++': 100.0, 'ketepatan_krama_alus': 0.7407}, 'meleset (ref Krama)': {'chrF++': 55.24, 'ketepatan_krama_alus': 0.0}}, 'diminta_krama_alus': 53, 'bisa_diubah': 0, 'diubah': 0}


[  1355s] valid trig_gain {'score': 106.7, 'chrF++': 85.76, 'ketepatan_krama_alus': 0.6981, 'per_jenis': {'benar (ref Krama Alus)': {'chrF++': 100.0, 'ketepatan_krama_alus': 0.7407}, 'meleset (ref Krama)': {'chrF++': 70.66, 'ketepatan_krama_alus': 0.6538}}, 'diminta_krama_alus': 53, 'bisa_diubah': 25, 'diubah': 22}


[  1357s] valid trig {'score': 106.7, 'chrF++': 85.76, 'ketepatan_krama_alus': 0.6981, 'per_jenis': {'benar (ref Krama Alus)': {'chrF++': 100.0, 'ketepatan_krama_alus': 0.7407}, 'meleset (ref Krama)': {'chrF++': 70.66, 'ketepatan_krama_alus': 0.6538}}, 'diminta_krama_alus': 53, 'bisa_diubah': 25, 'diubah': 22}


[  1359s] valid gain {'score': 106.45, 'chrF++': 85.51, 'ketepatan_krama_alus': 0.6981, 'per_jenis': {'benar (ref Krama Alus)': {'chrF++': 99.52, 'ketepatan_krama_alus': 0.7407}, 'meleset (ref Krama)': {'chrF++': 70.66, 'ketepatan_krama_alus': 0.6538}}, 'diminta_krama_alus': 53, 'bisa_diubah': 25, 'diubah': 23}


[  1361s] valid all {'score': 105.85, 'chrF++': 84.91, 'ketepatan_krama_alus': 0.6981, 'per_jenis': {'benar (ref Krama Alus)': {'chrF++': 98.34, 'ketepatan_krama_alus': 0.7407}, 'meleset (ref Krama)': {'chrF++': 70.66, 'ketepatan_krama_alus': 0.6538}}, 'diminta_krama_alus': 53, 'bisa_diubah': 25, 'diubah': 25}


[  1361s] MODE TERPILIH (valid): trig_gain


In [4]:
# ------------------------------------------------------------------ evaluasi TEST
V2 = "kaggle/final_3lv_v2/output"
INPUTS = {"rerank2.0": ("pred_m2m_base_rerank2.0.csv", "control_m2m_base_rerank2.0.csv"),
          "beam4": ("pred_m2m_base_beam4.csv", "control_m2m_base_beam4.csv")}


def evaluate(pred, ctrl, mode, base_cls):
    hyp, st_p = postedit(pred["hyp"], pred["level"], mode)
    res = {"test": {}}
    for lv in LEVELS + ["all"]:
        m = (pred["level"] == lv).values if lv != "all" else np.ones(len(pred), bool)
        h, r = [x for x, k in zip(hyp, m) if k], pred.loc[m, "javanese"].tolist()
        res["test"][lv] = {"n": len(r), "BLEU": round(bleu(h, r), 2), "chrF++": round(chrf(h, r), 2)}
        if lv in ("all", "krama_alus"):
            res["test"][lv]["chrF++_CI95"] = boot_ci(h, r)
    out, st_c = postedit(ctrl["output"], ctrl["requested"], mode)
    p = classify(out); y = ctrl["requested"].map(LID).values
    conf = np.zeros((3, 3), int)
    for a, b in zip(y, p):
        conf[a, b] += 1
    ka = y == KA
    O = {(s, l): o for s, l, o in zip(ctrl["indonesian"], ctrl["requested"], out)}
    u = sorted(set(ctrl["indonesian"]))
    res["control"] = {
        "level_accuracy": round(float(np.trace(conf) / conf.sum()), 4),
        "per_level_accuracy": {l: round(float(conf[i, i] / conf[i].sum()), 4) for i, l in enumerate(LEVELS)},
        "confusion_rows_requested_cols_predicted": conf.tolist(),
        "identical_rate_krama=krama_alus": round(float(np.mean([O[(s, "krama")] == O[(s, "krama_alus")] for s in u])), 4),
        # dibanding baseline (juri yang sama): berapa permintaan Krama Alus yang jadi benar / jadi salah
        "krama_alus_diperbaiki": int(((base_cls != KA) & (p == KA) & ka).sum()),
        "krama_alus_dirusak": int(((base_cls == KA) & (p != KA) & ka).sum())}
    res["edit_pred"], res["edit_control"] = st_p, st_c
    return res, hyp, out, p


results, examples = {}, []
for name, (fp, fc) in INPUTS.items():
    pred = pd.read_csv(path(f"{V2}/{fp}")); ctrl = pd.read_csv(path(f"{V2}/{fc}"))
    assert len(pred) == len(te) and len(ctrl) == 3 * te["indonesian"].nunique()
    base_cls = classify(ctrl["output"])
    results[name] = {}
    for m in MODES:
        r, hyp, out, p = evaluate(pred, ctrl, m, base_cls)
        results[name][m] = r
        log(name, m, "TEST all", r["test"]["all"], "| KA", r["test"]["krama_alus"],
            "| ctrl", r["control"]["level_accuracy"], r["control"]["per_level_accuracy"],
            "| +", r["control"]["krama_alus_diperbaiki"], "-", r["control"]["krama_alus_dirusak"])
        if m == best_mode:
            pd.DataFrame({"indonesian": ctrl["indonesian"], "requested": ctrl["requested"], "output": out,
                          "classified": [LEVELS[q] for q in p]}).to_csv(f"{OUT}/control_{name}_pe_{m}.csv", index=False)
            pp = pred.copy(); pp["hyp"] = hyp; pp.to_csv(f"{OUT}/pred_{name}_pe_{m}.csv", index=False)
            for s, l, a, b, q0, q1 in zip(ctrl["indonesian"], ctrl["requested"], ctrl["output"], out, base_cls, p):
                if a != b:
                    examples.append({"input": name, "indonesian": s, "sebelum": a, "sesudah": b,
                                     "juri_sebelum": LEVELS[q0], "juri_sesudah": LEVELS[q1]})
pd.DataFrame(examples).to_csv(f"{OUT}/contoh_perubahan.csv", index=False)

summary = {"pertanyaan": "post-editing leksikon Krama -> Krama Alus pada output model demo v2",
           "leksikon_andal": len(SUB), "judge": judge_info, "reranker": rr_info, "device": dev,
           "valid_simulasi": valid_sel, "mode_terpilih_valid": best_mode, "test": results}
json.dump(summary, open(f"{OUT}/summary_postedit.json", "w"), indent=1, ensure_ascii=False)
log("SELESAI")


[  1391s] rerank2.0 none TEST all {'n': 176, 'BLEU': 33.17, 'chrF++': 58.62, 'chrF++_CI95': [56.18, 61.27]} | KA {'n': 27, 'BLEU': 34.93, 'chrF++': 61.45, 'chrF++_CI95': [54.9, 68.68]} | ctrl 0.8489 {'ngoko': 0.9867, 'krama': 0.9867, 'krama_alus': 0.5733} | + 0 - 0


[  1421s] rerank2.0 trig_gain TEST all {'n': 176, 'BLEU': 33.17, 'chrF++': 58.62, 'chrF++_CI95': [56.18, 61.27]} | KA {'n': 27, 'BLEU': 34.93, 'chrF++': 61.45, 'chrF++_CI95': [54.9, 68.68]} | ctrl 0.8489 {'ngoko': 0.9867, 'krama': 0.9867, 'krama_alus': 0.5733} | + 0 - 0


[  1452s] rerank2.0 trig TEST all {'n': 176, 'BLEU': 33.17, 'chrF++': 58.62, 'chrF++_CI95': [56.18, 61.27]} | KA {'n': 27, 'BLEU': 34.93, 'chrF++': 61.45, 'chrF++_CI95': [54.9, 68.68]} | ctrl 0.8489 {'ngoko': 0.9867, 'krama': 0.9867, 'krama_alus': 0.5733} | + 0 - 0


[  1482s] rerank2.0 gain TEST all {'n': 176, 'BLEU': 33.27, 'chrF++': 58.64, 'chrF++_CI95': [56.23, 61.29]} | KA {'n': 27, 'BLEU': 35.53, 'chrF++': 61.58, 'chrF++_CI95': [54.85, 68.92]} | ctrl 0.8489 {'ngoko': 0.9867, 'krama': 0.9867, 'krama_alus': 0.5733} | + 0 - 0


[  1512s] rerank2.0 all TEST all {'n': 176, 'BLEU': 33.27, 'chrF++': 58.64, 'chrF++_CI95': [56.23, 61.29]} | KA {'n': 27, 'BLEU': 35.53, 'chrF++': 61.58, 'chrF++_CI95': [54.85, 68.92]} | ctrl 0.8489 {'ngoko': 0.9867, 'krama': 0.9867, 'krama_alus': 0.5733} | + 0 - 0


[  1541s] beam4 none TEST all {'n': 176, 'BLEU': 33.13, 'chrF++': 58.13, 'chrF++_CI95': [55.73, 60.74]} | KA {'n': 27, 'BLEU': 33.63, 'chrF++': 58.89, 'chrF++_CI95': [51.97, 66.42]} | ctrl 0.8133 {'ngoko': 0.9733, 'krama': 0.9867, 'krama_alus': 0.48} | + 0 - 0


[  1571s] beam4 trig_gain TEST all {'n': 176, 'BLEU': 33.29, 'chrF++': 58.26, 'chrF++_CI95': [55.82, 60.91]} | KA {'n': 27, 'BLEU': 34.68, 'chrF++': 59.67, 'chrF++_CI95': [53.01, 67.2]} | ctrl 0.8133 {'ngoko': 0.9733, 'krama': 0.9867, 'krama_alus': 0.48} | + 0 - 0


[  1603s] beam4 trig TEST all {'n': 176, 'BLEU': 33.29, 'chrF++': 58.26, 'chrF++_CI95': [55.82, 60.91]} | KA {'n': 27, 'BLEU': 34.68, 'chrF++': 59.67, 'chrF++_CI95': [53.01, 67.2]} | ctrl 0.8133 {'ngoko': 0.9733, 'krama': 0.9867, 'krama_alus': 0.48} | + 0 - 0


[  1633s] beam4 gain TEST all {'n': 176, 'BLEU': 33.29, 'chrF++': 58.24, 'chrF++_CI95': [55.8, 60.91]} | KA {'n': 27, 'BLEU': 34.68, 'chrF++': 59.6, 'chrF++_CI95': [52.85, 67.2]} | ctrl 0.8133 {'ngoko': 0.9733, 'krama': 0.9867, 'krama_alus': 0.48} | + 0 - 0


[  1663s] beam4 all TEST all {'n': 176, 'BLEU': 33.29, 'chrF++': 58.24, 'chrF++_CI95': [55.8, 60.91]} | KA {'n': 27, 'BLEU': 34.68, 'chrF++': 59.6, 'chrF++_CI95': [52.85, 67.2]} | ctrl 0.8133 {'ngoko': 0.9733, 'krama': 0.9867, 'krama_alus': 0.48} | + 0 - 0


[  1663s] SELESAI


In [5]:
# ------------------------------------------------------------------ ringkasan
rows = []
for name, rs in results.items():
    for m, r in rs.items():
        c = r["control"]
        rows.append({"input": name, "mode": m + (" *" if m == best_mode else ""),
                     "chrF++": r["test"]["all"]["chrF++"], "BLEU": r["test"]["all"]["BLEU"],
                     "chrF++ KA": r["test"]["krama_alus"]["chrF++"], "ketepatan": c["level_accuracy"],
                     "Krama": c["per_level_accuracy"]["krama"], "Krama Alus": c["per_level_accuracy"]["krama_alus"],
                     "diperbaiki": c["krama_alus_diperbaiki"], "dirusak": c["krama_alus_dirusak"],
                     "diubah": r["edit_control"]["diubah"]})
print("* = mode terpilih di valid")
pd.DataFrame(rows)


* = mode terpilih di valid


,input,mode,chrF++,BLEU,chrF++ KA,ketepatan,Krama,Krama Alus,diperbaiki,dirusak,diubah
0,rerank2.0,none,58.62,33.17,61.45,0.8489,0.9867,0.5733,0,0,0
1,rerank2.0,trig_gain *,58.62,33.17,61.45,0.8489,0.9867,0.5733,0,0,0
2,rerank2.0,trig,58.62,33.17,61.45,0.8489,0.9867,0.5733,0,0,0
3,rerank2.0,gain,58.64,33.27,61.58,0.8489,0.9867,0.5733,0,0,2
4,rerank2.0,all,58.64,33.27,61.58,0.8489,0.9867,0.5733,0,0,2
5,beam4,none,58.13,33.13,58.89,0.8133,0.9867,0.4800,0,0,0
6,beam4,trig_gain *,58.26,33.29,59.67,0.8133,0.9867,0.4800,0,0,2
7,beam4,trig,58.26,33.29,59.67,0.8133,0.9867,0.4800,0,0,2
8,beam4,gain,58.24,33.29,59.60,0.8133,0.9867,0.4800,0,0,3
9,beam4,all,58.24,33.29,59.60,0.8133,0.9867,0.4800,0,0,3
